# LLM Scoring - Coding Help!

## Import the required data bases

In [ ]:
import pandas as pd
import sqlalchemy as db

## Getting the files

In [ ]:
pmfeedback = pd.read_csv("https://github.com/casbdai/notebooks/raw/main/Module2/HypeCase/pmfeedback.csv")

In [ ]:
!wget https://github.com/casbdai/notebooks/raw/main/Module2/HypeCase/hypedb
engine = db.create_engine("sqlite:///hypedb")

In [ ]:
inspector = db.inspect(engine)
inspector.get_table_names()

## Applying LLM-Scoring

Run the following code to use a LLM for scoring:

In [ ]:
from openai import OpenAI
import json

In [ ]:
client = OpenAI(api_key="ENTER YOUR API KEY HERE")

Select the model and the reasoning effort that we want to use:

In [ ]:
model = "gpt-5.4"
# model = "gpt-4.1-mini"

# for gpt-5.4, select reasoning effort
reasoning_effort = "low" # alternatively try "medium", "high", or "none"

The following function sends texts to the OpenAI API and given a specific prompt such "Score the likelihood of a positive sentiment" it returns a numeric score reaching from 0 to 1. 0 reflects a low probability of positive sentiment and 1 a high propability.

In [ ]:
def score_text(text):

    try:
        request = {
            "model": model,
            "input": [
                {
                    "role": "system",
                    "content": (
                        prompt
                        + "Score the text with a likelihood between 0 (very low) and 1 (very high)."
                          "Return JSON only with the key 'score'."
                    ),
                },
                {
                    "role": "user",
                    "content": text[:400],
                },
            ],
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "score_response",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "score": {
                                "type": "number",
                                "minimum": 0,
                                "maximum": 1,
                            }
                        },
                        "required": ["score"],
                        "additionalProperties": False,
                    },
                    "strict": True,
                }
            },
        }

        if model.startswith("gpt-5.4"):
            request["reasoning"] = {"effort": reasoning_effort}

        response = client.responses.create(**request)

        data = json.loads(response.output_text)
        return data["score"]

    except Exception as e:
        print("An unexpected error occurred:", e)
        return None

To apply the code follow the following syntax:


In [ ]:
# dataframe["new_column_with_sentiment"] = dataframe["column_to_analyze"].apply(score_text)


As this code might take some time, you might want to track the progress. For this, use the tqdm library. It adds a .progress_apply function that helps you tracking how much time is needed to execute a function.

In [ ]:
from tqdm.auto import tqdm

tqdm.pandas()

# dataframe["new_column_with_sentiment"] = dataframe["column_to_analyze"].progress_apply(score_text)

## Few-shot prompting

Here is a pre-made function for you. Just select the model and the reasoning effort.

In [ ]:
model = "gpt-5.4"
# model = "gpt-4.1-mini"

# for gpt-5.4, select reasoning effort
reasoning_effort = "low" # alternatively try "medium", "high", or "none"

# Few-shot prompting with the same dataframe workflow as before
def score_text_few_shot(text, examples=None):

    if examples is None:
        examples = []

    messages = [
        {
            "role": "system",
            "content": (
                prompt
                + "Score the text with a likelihood between 0 and 1. "
                  "Return JSON only with the key 'score'."
            ),
        }
    ]

    for example in examples:
        messages.append(
            {"role": "user","content": example["input"],}
        )
        messages.append(
            {"role": "assistant","content": json.dumps(example["output"]),}
        )

    messages.append(
        {"role": "user","content": text[:400],}
    )

    try:
        request = {
            "model": model,
            "input": messages,
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "few_shot_score_response",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "score": {
                                "type": "number",
                                "minimum": 0,
                                "maximum": 1,
                            }
                        },
                        "required": ["score"],
                        "additionalProperties": False,
                    },
                    "strict": True,
                }
            },
        }

        if model.startswith("gpt-5.4"):
            request["reasoning"] = {"effort": reasoning_effort}

        response = client.responses.create(**request)
        return json.loads(response.output_text)["score"]

    except Exception as e:
        print("An unexpected error occurred:", e)
        return None

Apply it like this:

In [ ]:
prompt = "Score the likelihood of the text being _______."

example_1 = "_____"
score_1 = _.__
example_2 = "_____"
score_2 = _.__

few_shot_examples = [
    {
        "input": example_1,
        "output": {"score": score_1}
    },
    {
        "input": example_2,
        "output": {"score": score_2}
    },
]

# dataframe["new_column_with_sentiment"] = dataframe["column_to_analyze"].apply(score_text_few_shot, args=(few_shot_examples,))
